# 05.2 — LSTM from Scratch

**Problem it solves:** Vanilla RNN loses gradient signal over 10+ time steps. LSTMs maintain a separate cell state c_t that flows with additive (not multiplicative) updates — creating a gradient highway.

**The four gates:**
- **Forget gate:** How much of old cell state to keep: f_t = σ(W_f [h_{t-1}, x_t])
- **Input gate:** What new info to add: i_t = σ(W_i [h_{t-1}, x_t])
- **Cell update:** Candidate values: g_t = tanh(W_g [h_{t-1}, x_t])
- **Output gate:** What to expose as hidden state: o_t = σ(W_o [h_{t-1}, x_t])

**Cell state:** c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
**Hidden state:** h_t = o_t ⊙ tanh(c_t)

---

In [ ]:
import numpy as np

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-500,500)))

class LSTMCell:
    """
    Single LSTM cell — processes one time step.
    All four gates packed into one matrix multiplication for efficiency.
    """
    
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        H, D = hidden_size, input_size
        
        # Pack all gate weights into one matrix [4H x (D+H)]
        # Order: [forget, input, cell, output]
        scale = np.sqrt(2.0 / (D + H))
        self.W = np.random.randn(4*H, D+H) * scale
        self.b = np.zeros((4*H, 1))
        
        # Initialize forget gate bias to 1 (helps with long sequences)
        # Bias trick: makes the model default to remembering everything at start
        self.b[:H] = 1.0  # forget gate bias = 1
    
    def forward(self, x, h_prev, c_prev):
        """
        x:      input [D, 1]
        h_prev: previous hidden state [H, 1]
        c_prev: previous cell state [H, 1]
        Returns: h_next, c_next, cache
        """
        H = self.hidden_size
        
        # Concatenate input and previous hidden [D+H, 1]
        xh = np.vstack([x, h_prev])
        
        # All four gates in one matmul
        gates = self.W @ xh + self.b  # [4H, 1]
        
        f = sigmoid(gates[0:H])   # forget gate
        i = sigmoid(gates[H:2*H]) # input gate
        g = np.tanh(gates[2*H:3*H]) # cell gate (candidate values)
        o = sigmoid(gates[3*H:4*H]) # output gate
        
        # Cell state update: gradient highway!
        # dL/dc_prev = dL/dc_t × f_t
        # If f_t ≈ 1 (remember everything), gradient flows unchanged
        c_next = f * c_prev + i * g
        
        h_next = o * np.tanh(c_next)
        
        cache = (x, h_prev, c_prev, f, i, g, o, c_next, xh)
        return h_next, c_next, cache
    
    def backward(self, dh_next, dc_next, cache):
        """Backprop through one LSTM step."""
        H = self.hidden_size
        x, h_prev, c_prev, f, i, g, o, c_next, xh = cache
        
        # Gradient through output gate
        tanh_c = np.tanh(c_next)
        do = dh_next * tanh_c
        dc_next = dc_next + dh_next * o * (1 - tanh_c**2)
        
        # Gradient through cell state
        df = dc_next * c_prev
        di = dc_next * g
        dg = dc_next * i
        dc_prev = dc_next * f  # KEY: gradient flows through f gate!
        
        # Gradient through sigmoid/tanh of gates
        df_raw = df * f * (1 - f)    # sigmoid derivative
        di_raw = di * i * (1 - i)
        dg_raw = dg * (1 - g**2)     # tanh derivative
        do_raw = do * o * (1 - o)
        
        dgates = np.vstack([df_raw, di_raw, dg_raw, do_raw])  # [4H, 1]
        
        dW = dgates @ xh.T
        db = dgates
        dxh = self.W.T @ dgates
        
        dx = dxh[:self.input_size]
        dh_prev = dxh[self.input_size:]
        
        return dx, dh_prev, dc_prev, dW, db

# Test the cell
cell = LSTMCell(input_size=4, hidden_size=8)
x = np.random.randn(4, 1)
h = np.zeros((8, 1))
c = np.zeros((8, 1))

h2, c2, cache = cell.forward(x, h, c)
print(f"Input: {x.shape}, h_prev: {h.shape}, c_prev: {c.shape}")
print(f"h_next: {h2.shape}, c_next: {c2.shape}")
print(f"h_next range: [{h2.min():.3f}, {h2.max():.3f}]")
print(f"c_next range: [{c2.min():.3f}, {c2.max():.3f}]")

In [ ]:
# Gradient flow comparison: RNN vs LSTM

print("=== Gradient Flow: RNN vs LSTM ===")
print()

T = 50  # sequence length
H = 16

# RNN gradient: product of Jacobians → exponential decay
rnn_grads = [1.0]
W_hh = np.random.randn(H, H) * 0.1
spectral = np.max(np.abs(np.linalg.eigvals(W_hh)))
for t in range(T):
    rnn_grads.append(rnn_grads[-1] * spectral * 0.5)  # tanh shrinks by ~0.5

# LSTM gradient: additive cell state → near-constant gradient magnitude
lstm_grads = [1.0]
for t in range(T):
    # Forget gate ≈ 0.8 (remember 80% of previous cell state)
    lstm_grads.append(lstm_grads[-1] * 0.8)  # f_t controls this

print(f"{'Time step':12} {'RNN gradient':18} {'LSTM gradient':15}")
for t in [0, 5, 10, 20, 30, 50]:
    print(f"t={t:9d}  {rnn_grads[t]:18.6f}  {lstm_grads[t]:15.6f}")

print()
print("RNN gradient at t=50:", f"{rnn_grads[50]:.2e}", "(effectively zero)")
print("LSTM gradient at t=50:", f"{lstm_grads[50]:.4f}", "(still meaningful)")
print()
print("LSTM additive update: c_t = f_t * c_{t-1} + i_t * g_t")
print("Gradient dL/dc_{t-1} = dL/dc_t * f_t")
print("If f_t = 1: perfect gradient flow. If f_t = 0: gradient blocked (intentional forgetting)")

In [ ]:
# What do the gates actually do? Interpret them.

import re

# Sentiment analysis with LSTM (simplified: just show gate behavior)
cell = LSTMCell(input_size=5, hidden_size=4)
# Simulate processing 'not good' vs 'very good'

# Toy word embeddings
word_vecs = {
    'not':  np.array([[1.0, 0.0, -0.5, 0.2, 0.1]]).T,
    'very': np.array([[0.0, 1.0,  0.3, 0.1, 0.0]]).T,
    'good': np.array([[0.5, 0.5,  0.8, 0.3, 0.2]]).T,
    'bad':  np.array([[-0.5, -0.5, -0.8, -0.3, -0.2]]).T,
}

def process_sequence(words, cell):
    h = np.zeros((4, 1))
    c = np.zeros((4, 1))
    print(f"Sequence: {words}")
    for word in words:
        x = word_vecs.get(word, np.zeros((5,1)))
        h, c, cache = cell.forward(x, h, c)
        _, _, _, f, i, g, o, c_next, _ = cache
        print(f"  {word:6}  forget={f.mean():.3f}  input={i.mean():.3f}  "
              f"output={o.mean():.3f}  |h|={np.linalg.norm(h):.3f}")
    return h

h_not_good = process_sequence(['not', 'good'], cell)
print()
h_very_good = process_sequence(['very', 'good'], cell)
print()
print("Final hidden states differ -> different representations for 'not good' vs 'very good'")
print("(With trained weights, the forget gate would learn to 'remember' negation)")

## Summary

**LSTM key ideas:**
- Cell state = gradient highway with additive updates
- Forget gate = controlled forgetting (bias to 1 at init)
- Input gate = controlled writing
- Output gate = controlled reading

**The forget gate bias trick:** Initialize b_f = 1. This makes the model default to remembering everything. The model learns to forget selectively during training.

**What LSTMs can't do:**
- Still sequential (no parallelism)
- Still struggle with >100 step dependencies
- 4x more parameters than RNN for same hidden size
- Replaced by Transformers for most NLP tasks, but still used in streaming/real-time scenarios

---
**Next:** 05.3 — GRU (simpler, faster, nearly as good)